# Day 5 — Your First Useful Bedrock Build
### `intake_pipeline` (workflow)  ·  hand-rolled agent loop  ·  Strands version  ·  mocked TravelMind tools

**This notebook runs end to end with zero AWS credentials.**
It ships with a deterministic **mock** of the Bedrock Converse API that mimics the real response shape
(`output.message.content`, `usage`, `stopReason`, `toolUse` blocks). Every code path — workflow, hand-rolled
agent loop, and Strands — runs offline.

To run it against **real Bedrock**, set `USE_BEDROCK = True` in the config cell. The *same code* then calls the
real Converse API. Nothing else changes.

> Anchor case: **TravelMind**. Audience: mixed. Goal: you can rebuild this in your own project on Monday.

## How it works: one adapter, two modes

```
                       converse(messages, system, tool_config, ...)
                                        │
                 USE_BEDROCK ? ─────────┴───────── USE_BEDROCK = False (default)
                      │                                   │
                      ▼                                   ▼
        boto3 bedrock-runtime.converse()        _mock_converse()  ← deterministic, no creds
        (real model, real cost)                 (same response shape, runs anywhere)
```

**Prerequisites**
- Mock mode (default): Python 3.9+. No installs, no AWS.
- Real mode: `pip install boto3`, AWS credentials configured, Bedrock access in your region,
  and the Anthropic one-time usage form accepted once in the Bedrock console.
- Strands section (optional): `pip install strands-agents` **and** real Bedrock.

In [1]:
import json, re, itertools

# ============================ CONFIG ============================
USE_BEDROCK = False                                  # flip to True for the real model
REGION      = "us-east-1"
MODEL_ID    = "us.anthropic.claude-sonnet-4-6"       # cross-region inference profile (the us. prefix is required on-demand)

USAGE_LOG = []                                       # every call logs (input_tokens, output_tokens)

_client = None
def _bedrock_client():
    """Lazy boto3 client so the notebook runs with NO boto3 installed in mock mode."""
    global _client
    if _client is None:
        import boto3
        from botocore.config import Config
        _client = boto3.client(
            "bedrock-runtime", region_name=REGION,
            config=Config(retries={"max_attempts": 4, "mode": "adaptive"}),  # ThrottlingException is the #1 runtime failure
        )
    return _client

# ====================== THE ONE CALL ============================
def converse(messages, system=None, tool_config=None, inference_config=None, guardrail_config=None):
    """Single entry point. Real Bedrock when USE_BEDROCK, else the mock. Same response shape either way."""
    if USE_BEDROCK:
        kw = {"modelId": MODEL_ID, "messages": messages}
        if system:           kw["system"] = system
        if tool_config:      kw["toolConfig"] = tool_config
        if inference_config: kw["inferenceConfig"] = inference_config
        if guardrail_config: kw["guardrailConfig"] = guardrail_config
        resp = _bedrock_client().converse(**kw)
    else:
        resp = _mock_converse(messages, system, tool_config, inference_config)
    u = resp.get("usage", {})
    USAGE_LOG.append((u.get("inputTokens", 0), u.get("outputTokens", 0)))
    return resp

def cost_report(in_rate=3.0, out_rate=15.0):
    """Running token + cost ledger. Rates are ILLUSTRATIVE $/1M tokens — verify current Bedrock pricing."""
    ti = sum(a for a, _ in USAGE_LOG); to = sum(b for _, b in USAGE_LOG)
    cost = (ti / 1e6) * in_rate + (to / 1e6) * out_rate
    print(f"calls={len(USAGE_LOG)}  input_tokens={ti}  output_tokens={to}  est_cost=${cost:.4f}")
    print("(rates illustrative — confirm your model's price per 1M tokens before quoting numbers)")

print("config loaded · USE_BEDROCK =", USE_BEDROCK)


config loaded · USE_BEDROCK = False


In [2]:
# ===================================================================
#  MOCK ENGINE  —  the only reason this notebook runs without AWS.
#  It returns the SAME shape the real Converse API returns. Read it
#  once to understand the response structure, then forget it exists.
# ===================================================================
def _approx_tokens(s): return max(1, len(str(s)) // 4)
def _mock_usage(s):    o = _approx_tokens(s); return {"inputTokens": 120, "outputTokens": o, "totalTokens": 120 + o}
def _mock_text(text):
    return {"output": {"message": {"role": "assistant", "content": [{"text": text}]}},
            "stopReason": "end_turn", "usage": _mock_usage(text)}

def _user_text(messages):
    for m in messages:
        if m["role"] == "user":
            for b in m["content"]:
                if "text" in b: return b["text"]
    return ""

# --- workflow mocks (stand in for the LLM's language work) ---
_DEST = {"goa":"GOI","mumbai":"BOM","delhi":"DEL","bangalore":"BLR","bengaluru":"BLR",
         "chennai":"MAA","kochi":"COK","dubai":"DXB"}
def _mock_intake(text):
    t = text.lower()
    dest = next((c for k, c in _DEST.items() if k in t), None)
    pax  = 4 if "family" in t else (2 if any(w in t for w in ["wife","husband","me and","and i","couple","partner"]) else 1)
    cls  = "business" if "business" in t else ("first" if "first class" in t else "economy")
    flex = any(w in t for w in ["flexible","anytime","whenever","sometime"])
    dw   = next((w for w in ["next weekend","this weekend","next week","tomorrow","next month"] if w in t), None)
    return {"origin": None, "destination": dest, "date_window": dw, "pax": pax, "class": cls, "flexible": flex}
def _mock_classify(text):
    t = text.lower()
    if any(w in t for w in ["refund","complaint","angry","worst","terrible"]):       return "complaint"
    if any(w in t for w in ["reschedule","change my","modify"]):                      return "change"
    if any(w in t for w in ["book","fly","flight","travel","trip","ticket to","go to"]): return "booking"
    return "other"

# --- agent mock (drives the real loop: tool_use -> result -> ... -> end_turn) ---
_mock_ids = itertools.count(1)
def _tools_done(messages):
    id_to_name, done, results = {}, set(), {}
    for m in messages:
        if m["role"] == "assistant":
            for b in m.get("content", []):
                if "toolUse" in b: id_to_name[b["toolUse"]["toolUseId"]] = b["toolUse"]["name"]
        if m["role"] == "user":
            for b in m.get("content", []):
                if "toolResult" in b:
                    name = id_to_name.get(b["toolResult"]["toolUseId"])
                    if name:
                        done.add(name)
                        for c in b["toolResult"].get("content", []):
                            if "json" in c: results[name] = c["json"]
    return done, results
def _extract_fare_class(text):
    t = text.lower()
    for fc in ["saver economy","flexi economy","flexi","business","premium economy","first"]:
        if fc in t: return fc.title()
    return None
def _compose_answer(results):
    loy, fare, fl = results.get("check_loyalty_balance"), results.get("get_fare_rules"), results.get("search_flights")
    if fare and "error" in fare:
        return ("I can pull the rules, but I need your exact fare class — e.g. 'Saver Economy', 'Flexi', "
                "or 'Business'. Which fare did you book?")
    if fare:
        line = f"Your {fare.get('fare_class','fare')} ticket " + ("can be cancelled" if fare.get("cancellable") else "cannot be cancelled")
        if fare.get("fee") is not None: line += f", fee {fare['fee']}, refund {fare.get('refund','n/a')}"
        line += "."
        if loy: line = f"As a {loy['tier']} member: " + line + f" You have {loy['points']} points and could cover the fee in points."
        return line
    if fl:  r = fl["results"][0]; return f"Found {r['flight']} {r['origin']}->{r['destination']} on {r['date']} at {r['price']}."
    if loy: return f"You are {loy['tier']} tier with {loy['points']} points."
    return "Done. " + json.dumps(results)
def _mock_agent_turn(messages, tool_config):
    user, t = _user_text(messages), _user_text(messages).lower()
    needs = []
    if any(w in t for w in ["tier","points","loyalty","gold","silver","platinum","member","miles"]): needs.append("check_loyalty_balance")
    if any(w in t for w in ["cancel","change","refund","rules","reschedule"]):                         needs.append("get_fare_rules")
    if any(w in t for w in ["search","find","cheapest","options","available"]):                        needs.append("search_flights")
    if not needs: needs = ["get_fare_rules"]
    done, results = _tools_done(messages)
    for name in needs:
        if name not in done:
            if   name == "check_loyalty_balance": inp = {"member_id": "TM-9001"}
            elif name == "get_fare_rules":        inp = {"fare_class": _extract_fare_class(user) or "economy"}  # guesses if unstated -> shows a hallucinated arg
            else:                                 inp = {"origin": "BLR", "destination": "GOI", "date": "2026-06-13"}
            block = {"toolUse": {"toolUseId": f"tu_{next(_mock_ids)}", "name": name, "input": inp}}
            return {"output": {"message": {"role": "assistant", "content": [block]}},
                    "stopReason": "tool_use", "usage": _mock_usage(user)}
    return _mock_text(_compose_answer(results))

def _mock_converse(messages, system, tool_config, inference_config):
    if tool_config: return _mock_agent_turn(messages, tool_config)
    sys_text = " ".join(b.get("text", "") for b in (system or [])).lower()
    user = _user_text(messages)
    if "json" in sys_text and "travel" in sys_text: return _mock_text(json.dumps(_mock_intake(user)))
    if "classify" in sys_text:                       return _mock_text(_mock_classify(user))
    return _mock_text("Hello — I can read your message. (mock mode; set USE_BEDROCK=True for the real model.)")

print("mock engine ready")


mock engine ready


---
## Part A — The Workflow:  `intake_pipeline`

You own the steps; the model is one component. Messy free text in, structured booking intent out.

```
free text ─▶ classify(LLM) ─▶ booking? ──no──▶ {route_to: intent}        (stop — don't waste a call)
                                  │ yes
                                  ▼
                           to_intent(LLM) ─▶ enrich_airport_codes(python) ─▶ structured intent
```

Two of the three steps are LLM calls. The enrichment is plain Python — never pay a model to do a table lookup.

In [3]:
# ---- the three steps, then the pipeline ----
SYSTEM_INTAKE = [{"text":
    "You convert messy travel requests into JSON. "
    "Keys: origin, destination, date_window, pax, class, flexible. "
    "Use null for anything not stated. No prose. No code fences."}]

def to_intent(text):
    r = converse([{"role":"user","content":[{"text": text}]}],
                 system=SYSTEM_INTAKE, inference_config={"maxTokens":400, "temperature":0})  # 0 = repeatable
    raw = r["output"]["message"]["content"][0]["text"]
    try:    return json.loads(raw)
    except json.JSONDecodeError:
        return {"_error": "model did not return clean JSON", "_raw": raw}   # never trust clean JSON

SYSTEM_CLASSIFY = [{"text": "Classify the travel request as exactly one of: booking, change, complaint, other. "
                            "Reply with only the label."}]
def classify(text):
    r = converse([{"role":"user","content":[{"text": text}]}],
                 system=SYSTEM_CLASSIFY, inference_config={"maxTokens":10, "temperature":0})
    return r["output"]["message"]["content"][0]["text"].strip().lower()

AIRPORT_CITY = {"GOI":"Goa","BOM":"Mumbai","DEL":"Delhi","BLR":"Bengaluru","MAA":"Chennai","COK":"Kochi","DXB":"Dubai"}
def enrich_airport_codes(data):                 # plain python — NOT an LLM call
    if data.get("destination") in AIRPORT_CITY:
        data["destination_city"] = AIRPORT_CITY[data["destination"]]
    return data

def intake_pipeline(text):
    intent = classify(text)
    if intent != "booking":
        return {"route_to": intent}             # short-circuit: a complaint never reaches the extractor
    data = to_intent(text)
    return enrich_airport_codes(data)

print("workflow defined")


workflow defined


In [4]:
samples = [
    "hey so me and my wife wanna fly to goa sometime next weekend if its not too pricey, economy is fine, we're a bit flexible on dates",
    "I need to book business class to Dubai next week, just me",
    "this is the third time my refund hasn't come through, absolutely terrible service",
]
for s in samples:
    print("IN :", s[:70], "...")
    print("OUT:", json.dumps(intake_pipeline(s)))
    print("-" * 90)


IN : hey so me and my wife wanna fly to goa sometime next weekend if its no ...
OUT: {"origin": null, "destination": "GOI", "date_window": "next weekend", "pax": 2, "class": "economy", "flexible": true, "destination_city": "Goa"}
------------------------------------------------------------------------------------------
IN : I need to book business class to Dubai next week, just me ...
OUT: {"origin": null, "destination": "DXB", "date_window": "next week", "pax": 1, "class": "business", "flexible": false, "destination_city": "Dubai"}
------------------------------------------------------------------------------------------
IN : this is the third time my refund hasn't come through, absolutely terri ...
OUT: {"route_to": "complaint"}
------------------------------------------------------------------------------------------


---
## Part B — The Agent (hand-rolled loop)

Now the **model** decides which tools to call, and loops until the goal is met. We build the loop by hand once
so the framework (Part C) holds no mystery.

```
user ─▶ converse(messages, toolConfig)
           │
           ├─ stopReason == "end_turn"  ─▶  final answer   (stop)
           │
           └─ stopReason == "tool_use"
                  │   model returns toolUse{ name, input, toolUseId }
                  ▼
           YOUR CODE:  TOOLS[name](**input)            # validate args here — the model can hallucinate them
                  │   build toolResult{ toolUseId, json, status }
                  ▼
           append to messages  ─▶  loop   (bounded by max_iters — never `while True`)
```

**Read vs write.** Today's tools are all *read* tools. Notice nothing actually cancels a booking — a write/irreversible
action would sit behind a human checkpoint. That gap is deliberate.

In [5]:
# ---- mocked TravelMind tools (return canned data; the orchestration around them is real) ----
FARE_DB = {
    "Saver Economy": {"fare_class":"Saver Economy","cancellable":True,  "fee":2000, "refund":"partial", "change_fee":1500},
    "Flexi":         {"fare_class":"Flexi",         "cancellable":True,  "fee":0,    "refund":"full",    "change_fee":0},
    "Business":      {"fare_class":"Business",      "cancellable":True,  "fee":1000, "refund":"full",    "change_fee":500},
}
def get_fare_rules(fare_class):
    rec = FARE_DB.get(str(fare_class).title())
    return rec if rec else {"error": "unknown_fare_class", "got": fare_class, "valid": list(FARE_DB)}
def check_loyalty_balance(member_id="TM-9001"):
    return {"member_id": member_id, "tier": "Gold", "points": 18450}
def search_flights(origin, destination, date):
    return {"results": [{"flight": "6E-204", "origin": origin, "destination": destination, "date": date, "price": 5400}]}

TOOLS = {"get_fare_rules": get_fare_rules, "check_loyalty_balance": check_loyalty_balance, "search_flights": search_flights}

# ---- the tool schema sent to the model (Converse toolConfig) ----
TOOL_CONFIG = {"tools": [
    {"toolSpec": {"name": "get_fare_rules",
        "description": "Change, cancel and refund rules for a fare class. Call when the user asks to change or cancel a ticket.",
        "inputSchema": {"json": {"type": "object",
            "properties": {"fare_class": {"type": "string", "description": "e.g. 'Saver Economy', 'Flexi', 'Business'"}},
            "required": ["fare_class"]}}}},
    {"toolSpec": {"name": "check_loyalty_balance",
        "description": "Loyalty tier and points for a member. Call when tier, points or member benefits matter.",
        "inputSchema": {"json": {"type": "object",
            "properties": {"member_id": {"type": "string"}}, "required": []}}}},
    {"toolSpec": {"name": "search_flights",
        "description": "Find flights for a route and date. Call when the user wants flight options.",
        "inputSchema": {"json": {"type": "object",
            "properties": {"origin": {"type": "string"}, "destination": {"type": "string"}, "date": {"type": "string"}},
            "required": ["origin", "destination", "date"]}}}},
]}
print("tools + tool_config ready:", list(TOOLS))


tools + tool_config ready: ['get_fare_rules', 'check_loyalty_balance', 'search_flights']


In [6]:
def run_agent(user_msg, max_iters=6, verbose=True):
    messages = [{"role": "user", "content": [{"text": user_msg}]}]
    for i in range(max_iters):                                   # LOOP GUARD — the first line you write, not the last
        resp = converse(messages, tool_config=TOOL_CONFIG,
                        inference_config={"maxTokens": 600, "temperature": 0.2})
        out = resp["output"]["message"]
        messages.append(out)                                     # keep the model's turn in history
        if resp["stopReason"] != "tool_use":
            if verbose: print(f"  -> final answer after {i} tool round(s)")
            return out["content"][0]["text"]
        results = []
        for b in out["content"]:
            if "toolUse" in b:
                tu, name, args = b["toolUse"], b["toolUse"]["name"], b["toolUse"]["input"]
                if verbose: print(f"  [iter {i+1}] tool -> {name}({args})")
                try:
                    res = TOOLS[name](**args)                     # YOUR code runs the action
                    status = "error" if isinstance(res, dict) and "error" in res else "success"
                except Exception as e:
                    res, status = {"error": str(e)}, "error"      # do NOT swallow tool failures
                if verbose and status == "error": print(f"           ! tool error (surfaced, not swallowed): {res}")
                results.append({"toolResult": {"toolUseId": tu["toolUseId"], "content": [{"json": res}], "status": status}})
        messages.append({"role": "user", "content": results})     # hand results back, loop
    return "Stopped: hit max tool calls (loop guard fired)"

print("run_agent defined")


run_agent defined


In [7]:
# --- Q1: a real question that needs TWO tools, called in an order you did not specify ---
print(run_agent("I'm Gold tier — can I cancel my Saver Economy ticket, and what's the fee in points?"))


  [iter 1] tool -> check_loyalty_balance({'member_id': 'TM-9001'})
  [iter 2] tool -> get_fare_rules({'fare_class': 'Saver Economy'})
  -> final answer after 2 tool round(s)
As a Gold member: Your Saver Economy ticket can be cancelled, fee 2000, refund partial. You have 18450 points and could cover the fee in points.


In [8]:
# --- Q2: ambiguous input. Watch the model GUESS a fare class -> tool returns an error ->
#         the loop surfaces it and the agent asks a clarifying question instead of lying ---
print(run_agent("cancel my ticket"))


  [iter 1] tool -> get_fare_rules({'fare_class': 'economy'})
           ! tool error (surfaced, not swallowed): {'error': 'unknown_fare_class', 'got': 'economy', 'valid': ['Saver Economy', 'Flexi', 'Business']}
  -> final answer after 1 tool round(s)
I can pull the rules, but I need your exact fare class — e.g. 'Saver Economy', 'Flexi', or 'Business'. Which fare did you book?


### What Q2 just demonstrated

| Failure mode | Where you saw it | The guard that caught it |
|---|---|---|
| Hallucinated argument | model called `get_fare_rules(fare_class="economy")` — never stated | validate args / the tool's own `error` return |
| Silent tool failure | the tool returned `{"error": ...}` | we set `status="error"` and **did not swallow** it |
| Confident wrong answer | it *could* have said "Cancelled!" | read-only tools + a clarifying question instead of a fake outcome |
| Infinite loop | n/a here | `for _ in range(max_iters)` — never `while True` |

Every fix is cheap if you designed for it, expensive if you didn't.

---
## Part C — The Strands version (same agent, a fraction of the code)

Strands does the loop, the history, the tool plumbing and the guard for you. The `@tool` decorator infers the
schema from type hints + the docstring.

**Use the framework for almost everything. Hand-roll only when** you need custom logging, a non-standard runtime,
intervention between steps, or you are debugging why an agent misbehaves.

This cell needs `pip install strands-agents` **and** `USE_BEDROCK = True`. The tools reuse the **same mocked
TravelMind functions** — only the orchestration is real, so it stays a fair comparison to the hand-rolled loop.

In [9]:
try:
    from strands import Agent, tool
    _STRANDS = True
except ImportError:
    _STRANDS = False

if _STRANDS and USE_BEDROCK:
    @tool
    def fare_rules(fare_class: str) -> dict:
        "Change, cancel and refund rules for a fare class. Call when the user asks to change or cancel a ticket."
        return get_fare_rules(fare_class)           # reuse the mocked TravelMind data

    @tool
    def loyalty_balance(member_id: str = "TM-9001") -> dict:
        "Loyalty tier and points for a member."
        return check_loyalty_balance(member_id)

    agent = Agent(model=MODEL_ID, tools=[fare_rules, loyalty_balance])
    print(agent("I'm Gold tier — can I cancel my Saver Economy ticket, and cover the fee in points?"))
else:
    print("Strands demo skipped.")
    print("To run it:  pip install strands-agents   then set USE_BEDROCK = True (needs Bedrock access).")
    print("Same agent as Part B, but Strands owns the loop. Compare the line count to run_agent.")


Strands demo skipped.
To run it:  pip install strands-agents   then set USE_BEDROCK = True (needs Bedrock access).
Same agent as Part B, but Strands owns the loop. Compare the line count to run_agent.


---
## Part D — Guardrails + the cost ledger

**Never ship an agent naked.** A Bedrock Guardrail (created and versioned in the console) attaches as one extra
parameter on the *same* Converse call. PII masking, denied topics, content filters apply automatically.

```python
converse(messages, tool_config=TOOL_CONFIG,
         guardrail_config={"guardrailIdentifier": "tm-pii", "guardrailVersion": "1"})
```

(In mock mode the adapter accepts the parameter and ignores it — this just shows where it plugs in.
In real mode, enable cross-region on the guardrail if you use a cross-region inference profile.)

In [10]:
# the signature accepts a guardrail in both modes (mock ignores it):
_ = converse([{"role":"user","content":[{"text":"ping"}]}],
             guardrail_config={"guardrailIdentifier": "tm-pii", "guardrailVersion": "1"})
print("guardrail_config accepted by the same converse() call\n")

# the cost ledger across everything we ran today:
cost_report()


guardrail_config accepted by the same converse() call

calls=11  input_tokens=1320  output_tokens=194  est_cost=$0.0069
(rates illustrative — confirm your model's price per 1M tokens before quoting numbers)


---
## Wrap — take this back to your own work

**The decision, every time**
- Can you write the steps down?  → **Workflow.** Stop.
- Must the model choose actions at runtime?  → **Agent.** Tools small, loop guarded, actions gated, traced.
- Mixed?  → a **workflow with one bounded agentic step.** Most real systems.

**What today's build is still missing (each is a later session)**
memory across turns · model routing + fallback · real observability · evaluation · deployment + rollback · multi-agent coordination.

**To go live now:** set `USE_BEDROCK = True`, confirm your `MODEL_ID` and that it needs the `us.` inference profile,
accept the Anthropic usage form once, and read `usage` on every call so you know your cost before finance does.